In [ ]:
import re

import pandas as pd
from nltk.stem import PorterStemmer, WordNetLemmatizer

In [ ]:
hkhs = pd.read_parquet("../data/interim/hkhs.parquet")

#### Stemming and Lemmatisation

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def stem_desc(text):
    return " ".join(porter.stem(word) for word in clean_text(text).split())


def lemmatise_desc(text):
    return " ".join(
        lemmatiser.lemmatize(word) for word in clean_text(text).split()
    )

porter = PorterStemmer()
lemmatiser = WordNetLemmatizer()

In [ ]:
corpora = hkhs.query("field in ['description', 'commodity']")

corpora["desc_stem"] = corpora["desc"].apply(stem_desc)
corpora["desc_lemma"] = corpora["desc"].apply(lemmatise_desc)

In [ ]:
evaluation = hkhs.query("field == 'example'") \
    .groupby("desc", as_index=False) \
    .agg(
        chapter=("chapter", lambda x: sorted(set(x))),
        heading=("heading", lambda x: sorted(set(x))),
        subheading=("subheading", lambda x: sorted(set(x))),
    )

evaluation["desc_stem"] = evaluation["desc"].apply(stem_desc)
evaluation["desc_lemma"] = evaluation["desc"].apply(lemmatise_desc)

In [ ]:
corpora.to_parquet("../data/processed/corpora.parquet", index=False)
evaluation.to_parquet("../data/processed/evaluation.parquet", index=False)